In [1]:
import torch 
import math
import torch.nn as nn

## **Input Embedding layers**

In [2]:
##initialize the vocab size and its model dimension

class InputEmbeddings(nn.Module):
    ##initialize 
    def __init__(self, vocab_size: int, d_model:int)->None:
        super().__init__()
        # Set the model dimensionality and vocabulary size
        self.d_model = d_model
        self.vocab_size = vocab_size
        #forming Embedding Matrix with shape = (vocab_size, d_model)
        # Instantiate the embedding layer
        self.embeddings = nn.Embedding(vocab_size, d_model)
    
    ##forword feed
    def forward(self, x):
        # Return the embeddings multiplied by the square root of d_model
        ##scaled up factors to scale embeddings
        scaled_factor = math.sqrt(self.d_model)
        #dense vector representation
        tensor = self.embeddings(x)
        return tensor*scaled_factor

In [3]:
##define embedding layer 
embedding = InputEmbeddings(vocab_size=10000, d_model=512)
embedded_output = embedding(torch.tensor([[1,2,3,4], [5,6,7,8,]]))
print(embedded_output.shape)

torch.Size([2, 4, 512])


## **Positional Encoding layers**

In [6]:
##positional encoding layers
##define the positional encoding class by inherting from torch module
class PositionalEncoding(nn.Module):
    ##intialize
    def __init__(self, d_model:int, max_seq_length:int):
        super().__init__()
        # Create a matrix of zeros of dimensions max_seq_length by d_model
        ##initialize the positional embeddings (pe) to zeros
        pe = torch.zeros(max_seq_length, d_model)
        ##create the tensor of positions for each token in the sequence then it transform using unsqueeze
        ##so that it can be used in positional encoding calculations. unsqueeze(1)- converts into tensor.
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        ##division term in sin and cosine function
        div_term = torch.exp(torch.arange(0,d_model, 2, dtype=torch.float)* -(math.log(10000.0)/d_model))

        ##sin and cosine calculation
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        ##store pe without making learnable parameter during training
        self.register_buffer('pe', pe.unsqueeze(0))

    ##adding input token embeddings (X_input_token) with positional embedding (pe)
    def forward(self, x_embed_token):
        ##operation: element-wise addition
        #x_input_token + positional_encoding
        #token_representation + position_representation
        return x_embed_token+self.pe[:, :x_embed_token.size(1)]

In [7]:
pse_layer = PositionalEncoding(d_model =512, max_seq_length=4)
pse_layer_out = pse_layer(embedded_output)
print(pse_layer_out.shape)

torch.Size([2, 4, 512])


In [ ]:
#test
torch.arange(0,45).unsqueeze(0)

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
         18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
         36, 37, 38, 39, 40, 41, 42, 43, 44]])

In [18]:
##concepts

batch_size = 2
sequence_length = 4
d_model = 8

# simulate embedding output
x_input_token = torch.randn(batch_size, sequence_length, d_model)

print(x_input_token.shape)
print(x_input_token.size(1))

torch.Size([2, 4, 8])
4
